<a href="https://colab.research.google.com/github/ShilpiAg1988/ANN-Classification/blob/main/micrograd_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# micrograd exercises

1. watch the [micrograd video](https://www.youtube.com/watch?v=VMj-3S1tku0) on YouTube
2. come back and complete these exercises to level up :)

## section 1: derivatives

In [3]:
# here is a mathematical expression that takes 3 inputs and produces one output
from math import sin, cos

def f(a, b, c):
  return -a**3 + sin(3*b) - 1.0/c + b**2.5 - a**0.5

print(f(2, 3, 4))

6.336362190988558


In [4]:
# write the function df that returns the analytical gradient of f
# i.e. use your skills from calculus to take the derivative, then implement the formula
# if you do not calculus then feel free to ask wolframalpha, e.g.:
# https://www.wolframalpha.com/input?i=d%2Fda%28sin%283*a%29%29%29
from math import cos
def gradf(a, b, c):
  df_da=-3*a**2-.5*a**-.5
  df_db=3*cos(3*b)+2.5*b**1.5
  df_dc=c**-2
  return [df_da, df_db, df_dc] # todo, return [df/da, df/db, df/dc]

# expected answer is the list of
ans = [-12.353553390593273, 10.25699027111255, 0.0625]
yours = gradf(2, 3, 4)
for dim in range(3):
  ok = 'OK' if abs(yours[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {yours[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353553390593273
OK for dim 1: expected 10.25699027111255, yours returns 10.25699027111255
OK for dim 2: expected 0.0625, yours returns 0.0625


In [16]:
# now estimate the gradient numerically without any calculus, using
# the approximation we used in the video.
# you should not call the function df from the last cell

# ------------a**3 + sin(3*b) - 1.0/c + b**2.5 - a**0.5

h = .00000001
a, b, c = 2,3,4
da_df=(f(a+h, b, c) - f(a, b, c)) / h
db_df=(f(a, b+h, c) - f(a, b, c)) / h
dc_df=(f(a, b, c+h) - f(a, b, c)) / h
print(da_df)
print(db_df)
print(dc_df)
numerical_grad = [da_df, db_df, dc_df] # TODO
# -----------

for dim in range(3):
  ok = 'OK' if abs(numerical_grad[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad[dim]}")


-12.353553380251014
10.256990368162633
0.0624999607623522
OK for dim 0: expected -12.353553390593273, yours returns -12.353553380251014
OK for dim 1: expected 10.25699027111255, yours returns 10.256990368162633
OK for dim 2: expected 0.0625, yours returns 0.0624999607623522


In [18]:
# there is an alternative formula that provides a much better numerical
# approximation to the derivative of a function.
# learn about it here: https://en.wikipedia.org/wiki/Symmetric_derivative
# implement it. confirm that for the same step size h this version gives a
# better approximation.

# -----------
a, b, c = 2,3,4
da_df=(f(a+h, b, c) - f(a-h, b, c)) / (2*h)
db_df=(f(a, b+h, c) - f(a, b-h, c)) / (2*h)
dc_df=(f(a, b, c+h) - f(a, b, c-h)) / (2*h)
print(da_df)
print(db_df)
print(dc_df)
numerical_grad2 = [da_df, db_df, dc_df]
# -----------

for dim in range(3):
  ok = 'OK' if abs(numerical_grad2[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad2[dim]}")


-12.353553291433172
10.256990368162633
0.0624999607623522
OK for dim 0: expected -12.353553390593273, yours returns -12.353553291433172
OK for dim 1: expected 10.25699027111255, yours returns 10.256990368162633
OK for dim 2: expected 0.0625, yours returns 0.0624999607623522


## section 2: support for softmax

In [7]:
# Value class starter code, with many functions taken out
from math import exp, log

class Value:

  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data})"

  def __add__(self, other): # exactly as in the video
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')

    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward

    return out

  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')

    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
    return out

  def __pow__(self, other):
    assert isinstance(other, (int,float))
    out  = Value(self.data**other,(self,),'pow')

    def _backward():
      self.grad += (other*(self.data**(other-1))) * out.grad
    out._backward = _backward
    return out

  def exp(self):
    x = self.data
    out = Value(exp(self.data),(self,),'exp')

    def _backward():
      self.grad+= out.data * out.grad
    out._backward = _backward # Fix: Assign the _backward function

    return out

  def log(self):
    out= Value( log(self.data),(self,),'log')

    def _backward():
      self.grad+=(1/self.data) * out.grad
    out._backward=_backward

    return out

  def __neg__(self):
    return self * -1

  def __radd__(self, other): # other + self
    return self + other

  def __truediv__(self, other): # self / other
    return self * other**-1

  # ------
  # re-implement all the other functions needed for the exercises below
  # your code here
  # TODO
  # ------

  def backward(self): # exactly as in video
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)

    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

In [74]:
# without referencing our code/video __too__ much, make this cell work
# you'll have to implement (in some cases re-implemented) a number of functions
# of the Value object, similar to what we've seen in the video.
# instead of the squared error loss this implements the negative log likelihood
# loss, which is very often used in classification.

# this is the softmax function
# https://en.wikipedia.org/wiki/Softmax_function
def softmax(logits):
  # Calculate the exponentiated values as Value objects
  exps = [logit.exp() for logit in logits]

  # Sum all 'exp' Value objects to create a Value object for the denominator.
  # This ensures the sum is part of the computational graph, allowing gradients to propagate.
  sum_exps_val = exps[0]
  for i in range(1, len(exps)):
    sum_exps_val = sum_exps_val + exps[i]

  # Now perform the division using the Value object for the denominator
  out = [e / sum_exps_val for e in exps]
  return out

# this is the negative log likelihood loss function, pervasive in classification
logits = [Value(0.0), Value(3.0), Value(-2.0), Value(1.0)]

probs = softmax(logits)
print (probs)
# The line below also requires the Value class (in cell nAPe_RVrCTeO) to have
# __neg__ implemented for -probs[3].
loss = -probs[3].log() # dim 3 acts as the label for this input example
loss.backward()
print(loss.data)

ans = [0.041772570515350445, 0.8390245074625319, 0.005653302662216329, -0.8864503806400986]
for dim in range(4):
  ok = 'OK' if abs(logits[dim].grad - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {logits[dim].grad}")


[Value(data=0.04177257051535045), Value(data=0.839024507462532), Value(data=0.00565330266221633), Value(data=0.11354961935990122)]
2.1755153626167147
OK for dim 0: expected 0.041772570515350445, yours returns 0.041772570515350445
OK for dim 1: expected 0.8390245074625319, yours returns 0.8390245074625319
OK for dim 2: expected 0.005653302662216329, yours returns 0.005653302662216329
OK for dim 3: expected -0.8864503806400986, yours returns -0.8864503806400986


In [18]:
# verify the gradient using the torch library
# torch should give you the exact same gradient
import torch
#logits = [Value(0.0), Value(3.0), Value(-2.0), Value(1.0)]
tensiors = torch.tensor([l.data for l in logits], requires_grad=True)
tensors_exp = torch.exp(tensiors)
tens_sum= torch.sum(tensors_exp)
probs_torch = tensors_exp / tens_sum

#loss
loss = -torch.log(probs_torch[3])
loss.backward()

print('--Torch Gradient----')
for dim in range(len(tensiors)):
  print(tensiors.grad[dim])

print('--Custom Gradient----')
for dim in range(len(logits)):
  print(logits[dim].grad)
# Compare the gradients
print("\n--- Comparison ---")
ans = [0.041772570515350445, 0.8390245074625319, 0.005653302662216329, -0.8864503806400986]
for dim in range(len(tensiors)):
    torch_grad = tensiors.grad[dim].item()
    custom_grad = logits[dim].grad
    expected_grad = ans[dim] # Using the reference answer for comparison
    ok = 'OK' if abs(torch_grad - expected_grad) < 1e-5 else 'Wrong'
    print(f"{ok} for {dim} torch_grad {torch_grad} custom_grad {custom_grad} expected_grad {expected_grad}")




--Torch Gradient----
tensor(0.0418)
tensor(0.8390)
tensor(0.0057)
tensor(-0.8865)
--Custom Gradient----
0.041772570515350445
0.8390245074625319
0.005653302662216329
-0.8864503806400986

--- Comparison ---
OK for 0 torch_grad 0.041772566735744476 custom_grad 0.041772570515350445 expected_grad 0.041772570515350445
OK for 1 torch_grad 0.8390244245529175 custom_grad 0.8390245074625319 expected_grad 0.8390245074625319
OK for 2 torch_grad 0.005653302185237408 custom_grad 0.005653302662216329 expected_grad 0.005653302662216329
OK for 3 torch_grad -0.8864503502845764 custom_grad -0.8864503806400986 expected_grad -0.8864503806400986
